# Démonstration de séparation audio par ICA

Ce notebook reproduit l'expérience du rapport **Independent Component Analysis for Signal Separation**. Il charge trois enregistrements mixtes, applique **FastICA**, puis exporte trois composantes indépendantes au format WAV.

## Objectifs

- comprendre le modèle de mélange linéaire $X = AS$ ;
- vérifier que les trois observations sont compatibles ;
- séparer les sources avec `sklearn.decomposition.FastICA` ;
- écouter, visualiser et sauvegarder les résultats ;
- évaluer l'indépendance et l'erreur de reconstruction.

## 1. Principe de l'ICA

On suppose que les observations $X$ sont des combinaisons linéaires de sources inconnues $S$ :

$$X = AS$$

où $A$ est une matrice de mélange inconnue. ICA estime une matrice de séparation $W$ telle que :

$$Y = WX \approx S$$

Les hypothèses principales sont : sources statistiquement indépendantes, majoritairement non gaussiennes, mélange linéaire et nombre d'observations au moins égal au nombre de sources. L'ordre, le signe et l'amplitude des composantes retrouvées sont indéterminés : c'est une propriété normale de l'ICA.

## 2. Prérequis

Depuis un terminal placé dans ce dossier, installer les dépendances avec :

```bash
python -m pip install -r requirements.txt
```

Les entrées attendues sont `data/mixtures/ICA mix 1.wav`, `ICA mix 2.wav` et `ICA mix 3.wav`. Aucun FFmpeg ni `pydub` n'est nécessaire.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio, display
from scipy.io import wavfile
from scipy.stats import kurtosis
from sklearn.decomposition import FastICA

INPUT_DIR = Path("data/mixtures")
OUTPUT_DIR = Path("outputs/separated")
OUTPUT_DIR.mkdir(exist_ok=True)

AUDIO_FILES = [INPUT_DIR / f"ICA mix {index}.wav" for index in range(1, 4)]
missing = [str(path) for path in AUDIO_FILES if not path.exists()]
if missing:
    raise FileNotFoundError(f"Fichiers audio introuvables : {missing}")

print("Fichiers détectés :")
for path in AUDIO_FILES:
    print(f"- {path}")

## 3. Chargement et validation des mélanges

Les fichiers doivent avoir la même fréquence d'échantillonnage. Si leurs longueurs diffèrent, ils sont alignés sur le plus court. Les entiers PCM sont convertis dans l'intervalle $[-1, 1]$.

In [ ]:
def load_wav_mono(path):
    sample_rate, signal = wavfile.read(path)
    if signal.ndim == 2:
        signal = signal.mean(axis=1)

    original_dtype = signal.dtype
    if np.issubdtype(original_dtype, np.integer):
        info = np.iinfo(original_dtype)
        scale = float(max(abs(info.min), info.max))
        signal = signal.astype(np.float64) / scale
    else:
        signal = signal.astype(np.float64)

    if not np.all(np.isfinite(signal)):
        raise ValueError(f"Le fichier {path} contient des valeurs non finies.")
    return sample_rate, signal

loaded = [load_wav_mono(path) for path in AUDIO_FILES]
sample_rates = [item[0] for item in loaded]
if len(set(sample_rates)) != 1:
    raise ValueError(f"Fréquences d'échantillonnage différentes : {sample_rates}")

sample_rate = sample_rates[0]
min_length = min(len(item[1]) for item in loaded)
X = np.column_stack([item[1][:min_length] for item in loaded])
X = X - X.mean(axis=0, keepdims=True)
duration = min_length / sample_rate
time = np.arange(min_length) / sample_rate

print(f"Matrice X : {X.shape[0]} échantillons × {X.shape[1]} mélanges")
print(f"Fréquence : {sample_rate} Hz | Durée : {duration:.3f} s")
print(f"Rang numérique de X : {np.linalg.matrix_rank(X)}")

In [ ]:
for path in AUDIO_FILES:
    print(path.name)
    display(Audio(filename=str(path)))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 7), sharex=True)
for index, axis in enumerate(axes):
    axis.plot(time, X[:, index], linewidth=0.6)
    axis.set_ylabel(f"Mélange {index + 1}")
    axis.grid(alpha=0.2)
axes[-1].set_xlabel("Temps (s)")
fig.suptitle("Signaux observés")
plt.tight_layout()
plt.show()

print("Corrélation entre les mélanges :")
print(np.round(np.corrcoef(X, rowvar=False), 3))

## 4. Séparation avec FastICA

`whiten='unit-variance'` rend le comportement explicite avec les versions récentes de scikit-learn. Le `random_state` fixe rend l'expérience reproductible.

In [ ]:
ica = FastICA(
    n_components=3,
    whiten="unit-variance",
    algorithm="parallel",
    fun="logcosh",
    max_iter=2000,
    tol=1e-5,
    random_state=42,
)
S_estimated = ica.fit_transform(X)

print(f"Convergence en {ica.n_iter_} itération(s).")
print("Matrice de mélange estimée A :")
print(np.round(ica.mixing_, 4))
print("Matrice de séparation W :")
print(np.round(ica.components_, 4))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 7), sharex=True)
colors = ["#7b2cbf", "#2a9d8f", "#e76f51"]
for index, axis in enumerate(axes):
    axis.plot(time, S_estimated[:, index], color=colors[index], linewidth=0.6)
    axis.set_ylabel(f"Composante {index + 1}")
    axis.grid(alpha=0.2)
axes[-1].set_xlabel("Temps (s)")
fig.suptitle("Composantes indépendantes estimées")
plt.tight_layout()
plt.show()

## 5. Export WAV sans saturation

Chaque composante est normalisée séparément à 95 % de l'amplitude maximale du format `int16`. Cette méthode évite le facteur arbitraire `×100` du code historique, susceptible de provoquer un dépassement et une forte distorsion.

In [ ]:
def normalize_to_int16(signal, peak=0.95):
    max_abs = np.max(np.abs(signal))
    if max_abs == 0:
        return np.zeros_like(signal, dtype=np.int16)
    normalized = peak * signal / max_abs
    return np.round(normalized * np.iinfo(np.int16).max).astype(np.int16)

output_files = []
for index in range(S_estimated.shape[1]):
    output_path = OUTPUT_DIR / f"source_separee_{index + 1}.wav"
    wavfile.write(output_path, sample_rate, normalize_to_int16(S_estimated[:, index]))
    output_files.append(output_path)
    print(f"Créé : {output_path}")

In [ ]:
for path in output_files:
    print(path.name)
    display(Audio(filename=str(path)))

## 6. Évaluation

La corrélation linéaire proche de zéro est un contrôle utile, mais elle ne prouve pas à elle seule l'indépendance statistique. La kurtosis indique la non-gaussianité. Enfin, FastICA doit pouvoir reconstruire les mélanges avec une erreur numérique faible.

Comme les trois sources originales ne sont pas fournies dans ce dossier d'entrée, on ne peut pas calculer ici une corrélation source-à-source, un SDR ou un SI-SDR de référence. L'écoute reste donc une partie de l'évaluation.

In [ ]:
component_correlation = np.corrcoef(S_estimated, rowvar=False)
component_kurtosis = kurtosis(S_estimated, axis=0, fisher=True, bias=False)
X_reconstructed = ica.inverse_transform(S_estimated)
rmse = np.sqrt(np.mean((X - X_reconstructed) ** 2))
relative_rmse = rmse / np.sqrt(np.mean(X ** 2))

print("Corrélation entre les composantes :")
print(np.round(component_correlation, 4))
print("Kurtosis des composantes :", np.round(component_kurtosis, 3))
print(f"RMSE de reconstruction : {rmse:.3e}")
print(f"RMSE relative : {relative_rmse:.3e}")

## 7. Interprétation et limites

- Les trois fichiers sont trois **observations différentes** du même ensemble de sources ; trois copies d'un seul fichier ne suffiraient pas.
- Une composante peut être inversée, amplifiée ou apparaître dans un autre ordre sans que l'algorithme soit incorrect.
- ICA classique suppose un mélange instantané et linéaire. Les échos, réverbérations, décalages temporels et bruits réels rendent la séparation plus difficile.
- Une faible corrélation entre les sorties ne garantit pas une séparation perceptuelle parfaite : il faut également écouter les WAV exportés.
- Pour une évaluation quantitative complète, il faudrait disposer des trois sources originales utilisées pour fabriquer les mélanges.